In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [3]:
!pip install -q langchain-groq langchain-core requests

In [4]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [5]:
# tool create 

@tool #tool decorator
def multiply(a: int, b: int) -> int:
    """Given 2 numbers a and b this tool returns their product"""
    return a * b

In [6]:
print(multiply.invoke({'a':3 , 'b':4}))

12


In [7]:
multiply.name

'multiply'

In [8]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [9]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [10]:
# tool binding


In [11]:
llm = ChatGroq(model = 'openai/gpt-oss-20b')

In [12]:
llm_with_tools =  llm.bind_tools([multiply]) #a process where you connect tool with a given llm.All llm doesn't support tool binding

In [13]:
llm_with_tools.invoke('Hi how are you ?')

AIMessage(content='Hello! I’m just a bundle of code, so I don’t have feelings, but I’m here and ready to help you. How can I assist you today?', additional_kwargs={'reasoning_content': 'User: "Hi how are you ?" Simple greeting. We respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 134, 'total_tokens': 193, 'completion_time': 0.083818472, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.007652185, 'prompt_tokens_details': None, 'queue_time': 0.210826534, 'total_time': 0.091470657}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e23fc997ca', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a11d30-73dc-7bb1-adf1-1af39cb131dd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 134, 'output_tokens': 59, 'total_tokens': 193, 'output_token_details': {'reasoning': 16}})

In [14]:
query = HumanMessage('can you multiply 3 with 10?')

In [15]:
messages = [query]

In [16]:
messages

[HumanMessage(content='can you multiply 3 with 10?', additional_kwargs={}, response_metadata={})]

In [17]:
result = llm_with_tools.invoke(messages) #directly fetching tool calls

In [18]:
messages.append(result)

In [19]:
messages

[HumanMessage(content='can you multiply 3 with 10?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the multiply function.', 'tool_calls': [{'id': 'fc_6b86190e-0d78-4197-ba12-e1006229c45c', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 138, 'total_tokens': 173, 'completion_time': 0.035941943, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.006640823, 'prompt_tokens_details': None, 'queue_time': 0.386899672, 'total_time': 0.042582766}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_816fc87d18', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a11d30-75c6-7bf1-b065-fd5c314839c4-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'fc_6b86190e-0d78-4197-ba12-e1006229c45c', 

In [20]:
tool_result = multiply.invoke(result.tool_calls[0]) #tool calling
tool_result

ToolMessage(content='30', name='multiply', tool_call_id='fc_6b86190e-0d78-4197-ba12-e1006229c45c')

In [21]:
messages.append(tool_result)

In [22]:
messages

[HumanMessage(content='can you multiply 3 with 10?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the multiply function.', 'tool_calls': [{'id': 'fc_6b86190e-0d78-4197-ba12-e1006229c45c', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 138, 'total_tokens': 173, 'completion_time': 0.035941943, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.006640823, 'prompt_tokens_details': None, 'queue_time': 0.386899672, 'total_time': 0.042582766}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_816fc87d18', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a11d30-75c6-7bf1-b065-fd5c314839c4-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'fc_6b86190e-0d78-4197-ba12-e1006229c45c', 

In [23]:
llm_with_tools.invoke(messages).content

'30'

In [24]:
# tool create 
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool 
def get_conversion_factor(base_currency:str,target_currency:str) -> float:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency  
    
    """
    
    url = f'https://v6.exchangerate-api.com/v6/6ae44b882b5a2c67dc9351a4/pair/{base_currency}/{target_currency}'
    
    response = requests.get(url)
    return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float,InjectedToolArg]) -> float: #LLM do not fill this argument,I(developer/runtime) will inject theis value after running earlier tools
    """
    given a currency conversion rate this function calculates the target currency value from a given base currency value
    
    """
    
    return base_currency_value * conversion_rate
    
    

In [25]:
get_conversion_factor.invoke({'base_currency': 'USD','target_currency': 'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1791417601,
 'time_last_update_utc': 'Thu, 08 Oct 2026 00:00:01 +0000',
 'time_next_update_unix': 1791504001,
 'time_next_update_utc': 'Fri, 09 Oct 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 96.8339}

In [26]:
convert.invoke({'base_currency_value' : 10 , 'conversion_rate' :96.8339 })

968.3389999999999

In [27]:
# tool binding

llm = ChatGroq(model = "openai/gpt-oss-20b")

In [28]:
llm_with_tools = llm.bind_tools([get_conversion_factor,convert])

In [29]:
messages = [HumanMessage('What is the conversion factor between INR and USD,and based on that can you convert 10 INR to USD?')]

In [30]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD,and based on that can you convert 10 INR to USD?', additional_kwargs={}, response_metadata={})]

In [31]:
ai_message = llm_with_tools.invoke(messages)

In [32]:
messages.append(ai_message)

In [33]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'fc_a3d5c1e7-37d2-4fbf-8d35-9c028b3d74ff',
  'type': 'tool_call'}]

In [34]:
import json
for tool_call in ai_message.tool_calls:
    #print(tool_call)
    # execute the 1st tool and get the value of conversion rate
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        print(tool_message1)
        #fetch this conversion rate
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
     #append this tool message to messages list
        messages.append(tool_message1)
    # execute the 2nd tool using the conversion rate from tool 1
    
    if tool_call['name'] == 'convert' :
        # fetch the current argument 
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)
        
    
    

content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1791417601, "time_last_update_utc": "Thu, 08 Oct 2026 00:00:01 +0000", "time_next_update_unix": 1791504001, "time_next_update_utc": "Fri, 09 Oct 2026 00:00:01 +0000", "base_code": "INR", "target_code": "USD", "conversion_rate": 0.01033}' name='get_conversion_factor' tool_call_id='fc_a3d5c1e7-37d2-4fbf-8d35-9c028b3d74ff'


In [36]:
from IPython.display import display, Markdown
response = llm_with_tools.invoke(messages)

display(Markdown(response.content))

The current conversion factor from Indian Rupee (INR) to US Dollar (USD) is:

**1 INR → 0.01033 USD**

So, converting 10 INR:

\[
10 \text{ INR} \times 0.01033 \frac{\text{USD}}{\text{INR}} = 0.1033 \text{ USD}
\]

**Result:** 10 INR ≈ **$0.10 USD** (rounded to two decimal places).